# 02.2 — Auditoría de datos agrícolas EVA crudos

Auditoría de solo lectura equivalente al paso 02 climático. Revisa esquema, cobertura, llaves, códigos DANE, períodos, taxonomía y coherencia aritmética sin corregir `eva_cruda`.

In [ ]:
from pathlib import Path
import subprocess, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
REPO_REF = 'feature/SCRUM-15'
REPO_DIR = Path('/content/suelosabio') if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, 'https://github.com/cybercolombia/suelosabio.git', str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', 'fetch', 'origin', REPO_REF], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=REPO_DIR, check=True)
PIPELINE_DIR = REPO_DIR / 'notebooks' / 'ClimatePipeline'
if not PIPELINE_DIR.exists():
    PIPELINE_DIR = next((p for p in [Path.cwd(), Path.cwd() / 'ClimatePipeline', Path.cwd().parent / 'ClimatePipeline'] if (p / 'DatasetConfig.py').exists()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('No se encontró ClimatePipeline.')
sys.path.insert(0, str(PIPELINE_DIR)) if str(PIPELINE_DIR) not in sys.path else None
from DatasetConfig import cargar_configuracion_datasets
from ClimateProcessingUtils import escribir_parquet_atomico, escribir_json_atomico, escribir_texto_atomico
from CropYieldProcessing import AUDIT_VERSION, audit_raw_eva
import pandas as pd
from IPython.display import display
CONFIG = cargar_configuracion_datasets(in_colab=IN_COLAB)


In [ ]:
DATASET_ID = 'uejq-wxrr'
AUDITORIA_NOMBRE = 'eva_cruda_2019_2025_v1'
EJECUTAR_AUDITORIA = False
SOBRESCRIBIR_RESULTADOS = False
RAW_ROOT = CONFIG.eva_raw_root / f'fuente={DATASET_ID}'
OUTPUT_DIR = CONFIG.processed_root / 'auditorias_agricultura' / 'capa=eva_cruda' / f'fuente={DATASET_ID}' / f'auditoria={AUDITORIA_NOMBRE}'
archivos = sorted(RAW_ROOT.rglob('part-*.parquet')) if RAW_ROOT.exists() else []
print({'audit_version': AUDIT_VERSION, 'archivos': len(archivos), 'entrada': str(RAW_ROOT), 'salida': str(OUTPUT_DIR), 'ejecutar': EJECUTAR_AUDITORIA})

In [ ]:
resultado_auditoria = None
if not EJECUTAR_AUDITORIA:
    print('Auditoría desactivada. Revise rutas y active EJECUTAR_AUDITORIA.')
else:
    if not archivos:
        raise FileNotFoundError(f'No hay Parquet EVA en {RAW_ROOT}')
    crudo = pd.concat([pd.read_parquet(path) for path in archivos], ignore_index=True)
    resultado_auditoria = audit_raw_eva(crudo)
    tablas = {
        'resumen_auditoria.parquet': resultado_auditoria.summary,
        'nulos_columnas.parquet': resultado_auditoria.missingness,
        'llaves_duplicadas.parquet': resultado_auditoria.duplicate_keys,
        'banderas_calidad.parquet': resultado_auditoria.quality_flags,
        'cobertura_eva.parquet': resultado_auditoria.coverage,
    }
    for nombre, tabla in tablas.items():
        escribir_parquet_atomico(tabla, OUTPUT_DIR / nombre, SOBRESCRIBIR_RESULTADOS)
    resumen = resultado_auditoria.summary.iloc[0].to_dict()
    escribir_json_atomico({'estado': 'COMPLETA_CON_REVISION_PENDIENTE', 'audit_version': AUDIT_VERSION, 'dataset_id': DATASET_ID, 'archivos_entrada': len(archivos), **resumen}, OUTPUT_DIR / 'manifest.json', SOBRESCRIBIR_RESULTADOS)
    reporte = f'# Auditoría EVA cruda\n\n- Versión: `{AUDIT_VERSION}`\n- Filas: {resumen["filas"]:,}\n- Cultivos: {resumen["cultivos"]:,}\n- Municipios: {resumen["municipios"]:,}\n\n`COMPLETA_CON_REVISION_PENDIENTE` confirma que el diagnóstico terminó; no aprueba todavía la curación.\n'
    escribir_texto_atomico(reporte, OUTPUT_DIR / 'AuditoriaEvaCruda.md', SOBRESCRIBIR_RESULTADOS)
    display(resultado_auditoria.summary)
